# ISL Hand Shape Recognition — Full Training
**Steps:**
1. Upload your dataset zip files
2. Extract & prepare (10% per person/letter + augmentation)
3. Train MobileNetV3 with GPU (2-phase: frozen → fine-tune)
4. Validate & Test
5. Download trained model

**First: Runtime → Change runtime type → T4 GPU**

In [ ]:
# Check GPU is available
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 1. Upload Dataset
Upload all Person zip files (Person1.zip through Person6.zip) and the Person1 folder.

**Option A: Upload from local machine** (run the cell below)

**Option B: Google Drive** (skip to the Drive cell)

In [ ]:
# OPTION A: Upload zip files directly
# This will open a file picker — select all Person*.zip files
import os
from google.colab import files

os.makedirs('dataset/Frames', exist_ok=True)
print('Upload your Person*.zip files (Person1.zip through Person6.zip):')
uploaded = files.upload()
for fname in uploaded:
    os.rename(fname, f'dataset/Frames/{fname}')
    print(f'  Moved {fname} → dataset/Frames/{fname}')

In [ ]:
# OPTION B: Mount Google Drive (if you uploaded zips to Drive instead)
# Uncomment and edit the path below:

# from google.colab import drive
# drive.mount('/content/drive')
# !cp /content/drive/MyDrive/YOUR_FOLDER/Person*.zip dataset/Frames/

In [ ]:
# Extract all zip files
import zipfile
from pathlib import Path
from tqdm import tqdm

FRAMES_DIR = Path('dataset/Frames')

for zf_path in sorted(FRAMES_DIR.glob('Person*.zip')):
    person = zf_path.stem
    dest = FRAMES_DIR / person
    if dest.exists() and any(dest.iterdir()):
        print(f'{person}/ already extracted')
        continue
    dest.mkdir(exist_ok=True)
    print(f'Extracting {zf_path.name}...')
    with zipfile.ZipFile(zf_path) as z:
        for member in tqdm(z.namelist(), desc=person):
            fname = Path(member).name
            if fname.lower().endswith('.jpg'):
                target = dest / fname
                if not target.exists():
                    with z.open(member) as src, open(target, 'wb') as dst:
                        dst.write(src.read())

# Count
total = sum(1 for d in FRAMES_DIR.glob('Person*') if d.is_dir()
            for _ in d.glob('*.jpg'))
print(f'\nTotal frames extracted: {total:,}')

## 2. Prepare Dataset with Augmentation

In [ ]:
import re
import random
import shutil
from collections import defaultdict
import cv2
import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

FILENAME_RE = re.compile(r'^(Person\d+)-([A-Z])-(\d+)-(\d+)\.jpg$', re.IGNORECASE)
OUTPUT_DIR = Path('prepared_data')

# Build manifest: person → letter → [paths]
manifest = defaultdict(lambda: defaultdict(list))
for pdir in sorted(FRAMES_DIR.glob('Person*')):
    if not pdir.is_dir(): continue
    for fpath in pdir.iterdir():
        m = FILENAME_RE.match(fpath.name)
        if m:
            manifest[m.group(1)][m.group(2).upper()].append(str(fpath))

print(f'Persons: {sorted(manifest.keys())}')
print(f'Total frames: {sum(len(l) for p in manifest.values() for l in p.values()):,}')

def augment_image(img, rng):
    h, w = img.shape[:2]
    augs = []
    # Rotation
    M = cv2.getRotationMatrix2D((w/2, h/2), rng.uniform(-15, 15), 1)
    augs.append(cv2.warpAffine(img, M, (w, h), borderMode=cv2.BORDER_REFLECT_101))
    # Brightness/contrast
    augs.append(cv2.convertScaleAbs(img, alpha=rng.uniform(0.7, 1.3), beta=int(rng.integers(-30, 31))))
    # Flip
    augs.append(cv2.flip(img, 1))
    # Blur
    augs.append(cv2.GaussianBlur(img, (5, 5), 0))
    # Noise
    noise = rng.normal(0, 15, img.shape).astype(np.int16)
    augs.append(np.clip(img.astype(np.int16) + noise, 0, 255).astype(np.uint8))
    # Random crop
    cf = rng.uniform(0.8, 0.95)
    nh, nw = int(h*cf), int(w*cf)
    t, l = rng.integers(0, h-nh+1), rng.integers(0, w-nw+1)
    augs.append(cv2.resize(img[t:t+nh, l:l+nw], (w, h)))
    return augs

# ---- CHANGE THIS TO CONTROL DATA SIZE ----
SAMPLE_FRACTION = 10  # Use 1/N of data per person/letter. 10 = 10%, 5 = 20%, 1 = 100%
# -------------------------------------------

if OUTPUT_DIR.exists():
    shutil.rmtree(OUTPUT_DIR)

rng = np.random.default_rng(RANDOM_SEED)
total_orig, total_aug = 0, 0

for person in sorted(manifest.keys()):
    for letter in sorted(manifest[person].keys()):
        paths = manifest[person][letter]
        sample_size = max(1, len(paths) // SAMPLE_FRACTION)
        sampled = random.sample(paths, sample_size)
        
        out = OUTPUT_DIR / letter
        out.mkdir(parents=True, exist_ok=True)
        
        for orig_path in sampled:
            img = cv2.imread(orig_path)
            c = total_orig + total_aug
            cv2.imwrite(str(out / f'{person}_{c:06d}_orig.jpg'), img)
            total_orig += 1
            for aug_img in augment_image(img, rng):
                c = total_orig + total_aug
                cv2.imwrite(str(out / f'{person}_{c:06d}_aug.jpg'), aug_img)
                total_aug += 1

print(f'\nOriginals: {total_orig:,}')
print(f'Augmented: {total_aug:,}')
print(f'Total:     {total_orig + total_aug:,}')

## 3. Train the CNN (MobileNetV3-Small)
2-phase training:
- **Phase 1**: Frozen backbone, train classifier (3 epochs)
- **Phase 2**: Unfreeze last layers, fine-tune (12 epochs)

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader, random_split
from torchvision import transforms, models
from PIL import Image
from tqdm import tqdm

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
DATA_DIR = Path('prepared_data')

LETTERS = sorted([d.name for d in DATA_DIR.iterdir() if d.is_dir()])
LABEL_MAP = {l: i for i, l in enumerate(LETTERS)}
print(f'Device: {DEVICE}')
print(f'Classes: {len(LETTERS)}')

class ISLDataset(Dataset):
    def __init__(self, transform=None):
        self.samples = []
        self.transform = transform or transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor(),
            transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225]),
        ])
        for ld in DATA_DIR.iterdir():
            if not ld.is_dir(): continue
            label = LABEL_MAP[ld.name]
            for p in ld.glob('*.jpg'):
                self.samples.append((str(p), label))
    def __len__(self): return len(self.samples)
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        return self.transform(Image.open(path).convert('RGB')), label

dataset = ISLDataset()
total = len(dataset)
train_n = int(0.70 * total)
val_n = int(0.15 * total)
test_n = total - train_n - val_n
train_data, val_data, test_data = random_split(
    dataset, [train_n, val_n, test_n],
    generator=torch.Generator().manual_seed(42)
)

train_loader = DataLoader(train_data, batch_size=64, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_data, batch_size=64, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_data, batch_size=64, num_workers=2, pin_memory=True)

print(f'Train: {train_n:,}, Val: {val_n:,}, Test: {test_n:,}')

In [ ]:
# Create model
model = models.mobilenet_v3_small(weights='DEFAULT')
for p in model.features.parameters():
    p.requires_grad = False
in_feat = model.classifier[3].in_features
model.classifier[3] = nn.Linear(in_feat, len(LETTERS))
model = model.to(DEVICE)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
scaler = torch.amp.GradScaler('cuda')

def train_epoch(model, loader, criterion, optimizer):
    model.train()
    losses, correct, total = [], 0, 0
    for imgs, labels in tqdm(loader, desc='train', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            loss = criterion(logits, labels)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()
        losses.append(loss.item())
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return sum(losses)/len(losses), correct/total

@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    losses, correct, total = [], 0, 0
    for imgs, labels in tqdm(loader, desc='eval', leave=False):
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        with torch.amp.autocast('cuda'):
            logits = model(imgs)
            loss = criterion(logits, labels)
        losses.append(loss.item())
        correct += (logits.argmax(1) == labels).sum().item()
        total += labels.size(0)
    return sum(losses)/len(losses), correct/total

print('Model ready. Training next...')

In [ ]:
# ============ PHASE 1: Frozen backbone, train classifier ============
print('='*60)
print('PHASE 1: Training classifier (backbone frozen)')
print('='*60)

optimizer = optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()),
                        lr=1e-3, weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=3)
best_val_acc = 0

for epoch in range(1, 4):
    tl, ta = train_epoch(model, train_loader, criterion, optimizer)
    vl, va = eval_epoch(model, val_loader, criterion)
    print(f'Epoch {epoch}: train_acc={ta:.2%} val_acc={va:.2%} train_loss={tl:.3f} val_loss={vl:.3f}')
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  -> Saved (val_acc={va:.2%})')
    scheduler.step()

print(f'\nPhase 1 best val: {best_val_acc:.2%}')

In [ ]:
# ============ PHASE 2: Unfreeze last layers, fine-tune ============
print('='*60)
print('PHASE 2: Fine-tuning (last 4 backbone layers unfrozen)')
print('='*60)

# Unfreeze last 4 feature blocks
for layer in list(model.features.children())[-4:]:
    for p in layer.parameters():
        p.requires_grad = True

optimizer = optim.AdamW([
    {'params': model.features.parameters(), 'lr': 1e-4},
    {'params': model.classifier.parameters(), 'lr': 5e-4},
], weight_decay=1e-4)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=12)

for epoch in range(4, 16):
    tl, ta = train_epoch(model, train_loader, criterion, optimizer)
    vl, va = eval_epoch(model, val_loader, criterion)
    print(f'Epoch {epoch:2d}: train_acc={ta:.2%} val_acc={va:.2%} train_loss={tl:.3f} val_loss={vl:.3f}')
    if va > best_val_acc:
        best_val_acc = va
        torch.save(model.state_dict(), 'best_model.pth')
        print(f'  -> Saved (val_acc={va:.2%})')
    scheduler.step()

print(f'\nBest val accuracy: {best_val_acc:.2%}')

## 4. Test on Held-Out Data

In [ ]:
# Load best model and run final test
model.load_state_dict(torch.load('best_model.pth', map_location=DEVICE, weights_only=True))
test_loss, test_acc = eval_epoch(model, test_loader, criterion)

print('='*60)
print(f'FINAL TEST RESULTS')
print(f'  Best val accuracy:  {best_val_acc:.2%}')
print(f'  Test loss:          {test_loss:.3f}')
print(f'  Test accuracy:      {test_acc:.2%}')
print('='*60)

In [ ]:
# Per-letter accuracy breakdown
from collections import Counter

model.eval()
correct_per = Counter()
total_per = Counter()

with torch.no_grad():
    for imgs, labels in test_loader:
        imgs, labels = imgs.to(DEVICE), labels.to(DEVICE)
        preds = model(imgs).argmax(1)
        for pred, label in zip(preds, labels):
            l = LETTERS[label.item()]
            total_per[l] += 1
            if pred == label:
                correct_per[l] += 1

print(f'{"Letter":>6} {"Correct":>8} {"Total":>6} {"Acc":>8}')
print('-' * 32)
for l in LETTERS:
    t = total_per[l]
    c = correct_per[l]
    acc = c/t if t > 0 else 0
    print(f'{l:>6} {c:>8} {t:>6} {acc:>8.1%}')

## 5. Download Trained Model
Download `best_model.pth` and put it in your local `checkpoints/` folder.
Then run `python live_inference.py --model checkpoints/best_model.pth`

In [ ]:
from google.colab import files
files.download('best_model.pth')